<a href="https://colab.research.google.com/github/shashwatahalder/Spark-Code-Repo/blob/master/beamnotebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!{'pip install --quiet apache_beam'}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 960.0/960.0 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, whi

In [4]:
import apache_beam as beam
print(beam.__version__)

2.73.0


In [5]:
!{'mkdir -p data'}

In [6]:
from google.colab import files
uploaded = files.upload()

Saving dept_data.txt to dept_data.txt


In [7]:
!cat dept_data.txt

149633CM,Marco,10,Accounts,1-01-2019
212539MU,Rebekah,10,Accounts,1-01-2019
231555ZZ,Itoe,10,Accounts,1-01-2019
503996WI,Edouard,10,Accounts,1-01-2019
704275DC,Kyle,10,Accounts,1-01-2019
957149WC,Kyle,10,Accounts,1-01-2019
241316NX,Kumiko,10,Accounts,1-01-2019
796656IE,Gaston,10,Accounts,1-01-2019
331593PS,Beryl,20,HR,1-01-2019
560447WH,Olga,20,HR,1-01-2019
222997TJ,Leslie,20,HR,1-01-2019
171752SY,Mindy,20,HR,1-01-2019
153636AS,Vicky,20,HR,1-01-2019
745411HT,Richard,20,HR,1-01-2019
298464HN,Kirk,20,HR,1-01-2019
783950BW,Kaori,20,HR,1-01-2019
892691AR,Beryl,20,HR,1-01-2019
245668UZ,Oscar,20,HR,1-01-2019
231206QD,Kumiko,30,Finance,1-01-2019
357919KT,Wendy,30,Finance,1-01-2019
472418ZG,Cristobal,30,Finance,1-01-2019
442292OI,Erika,30,Finance,1-01-2019
503647MN,Sebastien,30,Finance,1-01-2019
245319LD,Valerie,30,Finance,1-01-2019
818776XR,Dolly,30,Finance,1-01-2019
259229ZU,Emily,30,Finance,1-01-2019
349360YC,Kaori,30,Finance,1-01-2019
951594MT,Hitomi,30,Finance,1-01-2019
149633CM,Marco,10,

In [19]:
! ls -lrt data

total 8
-rw-r--r-- 1 root root 36 May 20 18:54 output.txt-00000-of-00001
-rw-r--r-- 1 root root 16 May 20 18:58 outputnumbers.txt-00000-of-00001


In [11]:
! cat data/output.txt-00000-of-00001

This is Shashwata
Apache Beam
Hello


In [15]:
import apache_beam as beam

p1 = beam.Pipeline()

lines = (
    p1
    | beam.Create([
        'This is Shashwata',
        'Apache Beam',
        'Hello'
    ])
)

lines_write = ( lines | 'Write' >> beam.io.WriteToText('data/output.txt') )

p1.run()

!{'cat data/output.txt-00000-of-00001'}

This is Shashwata
Apache Beam
Hello


In [20]:
import apache_beam as beam

p2 = beam.Pipeline()

numbers = (p2 | beam.Create([1,2,3,4,4,5,6,7]) | beam.io.WriteToText('data/outputnumbers.txt'))

p2.run()

!{'cat data/outputnumbers.txt-00000-of-00001'}

1
2
3
4
4
5
6
7


In [27]:
import apache_beam as beam

def SplitRow(element):
  return element.split(',')

def filtering(element):
  return element[3] == 'Accounts'


p1 = beam.Pipeline()
attendance_count = (
    p1
    |beam.io.ReadFromText('dept_data.txt')
    |beam.Map(SplitRow) # FlatMap (lambda element : element.split(','))
    |beam.Filter(filtering) # (lambda element : element[3]=='Accounts')
    |beam.Map(lambda record: (record[1],1))
    |beam.CombinePerKey(sum)
    |beam.io.WriteToText('data/output_new')
)
p1.run()

!{('head -20 data/output_new-00000-of-00001')}

('Marco', 31)
('Rebekah', 31)
('Itoe', 31)
('Edouard', 31)
('Kyle', 62)
('Kumiko', 31)
('Gaston', 31)
('Ayumi', 30)
